# COMP5423 BrowseComp-Plus 智能体：系统实现与本地复测

本 Notebook 只围绕当前工作目标组织：**（1）系统实现；（2）本地复测**。代码实现位于项目 `src/` 和 `skills/`，Notebook 说明架构并提供本地演示、缓存/索引检查、验证集复测和输出检查。

模型 API key 只放在项目 `.env` 中。Notebook 的单题演示和批量复测会调用 DeepSeek API，产生费用；相关运行开关默认关闭。


## 1. 系统实现

### 1.1 系统目标与评分对应

系统在固定 BrowseComp-Plus 文档集合上回答多线索问题，结合检索、LLM 推理、工具调用和运行时加载的可复用技能。课程系统实现部分共 40%。

| 子项 | 分值 | 项目实现 |
|---|---:|---|
| 检索与工具设计 | 12% | BM25 稀疏检索及 BGE 密集检索；单一 `search` 检索工具；`open`、`find` 辅助工具 |
| 基础工具调用智能体 | 4% | DeepSeek V4 Flash function calling 选择工具并使用返回结果 |
| 迭代式智能体循环 | 5% | 后续搜索、打开、查找动作可依据前序观察调整 |
| 上下文管理 | 5% | 独立结构化状态、证据筛选和上下文裁剪 |
| 技能基本结构 | 6% | `skills/*/SKILL.md`，运行时加载 |
| 自动技能选择 bonus | 4% | 两个不同技能按问题特征或低证据状态自动选择 |
| 用户界面 | 4% | Streamlit 页面支持提问、查看答案、证据、技能和轨迹 |

检索方法类别评分为：一类 4 分、两类 6 分、三类 8 分；不同检索器必须通过**一个统一检索工具**暴露。当前实现有 BM25 和 BGE encoder dense；`dense_instruct` 只是 BGE 查询前缀，不能直接认定达到课程定义的 LLM 嵌入式指令检索第三类。工具评分要求统一检索工具加至少一种可调用辅助工具；报告需列出工具名称、输入输出和工具类型数。


## 2. 系统设计方法与关键组件

系统采用“检索器 + 有界工具调用智能体”。用户提交多线索问题后，DeepSeek V4 Flash 每轮选择一个结构化动作。BM25S 在固定语料上检索候选文档；智能体通过 `open` 读取文档全文，通过 `find` 定位短语，再基于打开过的证据给出答案和文档 ID 引用。检索分数、查询来源、打开状态、证据、轨迹和 token / 工具调用统计都会写入结果。

| 组件 | 作用 | 项目入口 |
|---|---|---|
| 数据加载 | 从 Hugging Face 数据集读取验证问题和固定语料；评测时可离线使用缓存 | `src/data.py` |
| 稀疏索引 | 在 100,195 篇文档上构建和载入 BM25S 索引 | `src/build_index.py`, `src/retrieval.py` |
| 可选向量检索 | BGE small embedding；支持 dense、instruction-aware dense 和 hybrid | `src/build_dense_index.py`, `src/dense.py` |
| 工具层 | 单查询 / 多查询检索、打开文档、短语查找、查询来源与排名记录 | `src/retrieval.py` |
| 智能体 | DeepSeek function calling、动作校验、技能选择、搜索恢复、上下文预算、证据引用与轨迹 | `src/agent.py`, `src/skills.py`, `skills/` |
| 评测与分析 | 生成预测 JSONL；计算 normalized EM、覆盖率、成本、停止原因及引用有效性 | `src/evaluate.py`, `src/analyze_results.py` |
| 用户界面 | Streamlit 交互式问题输入、答案、统计、轨迹和证据查看 | `src/app.py` |
| 本 Notebook | 将报告内容、历史实测结果和交互演示放在同一阅读 / 运行界面 | `COMP5423_SearchAgent.ipynb` |

API 密钥只从项目根目录的 `.env` 读取。预测文件和本 Notebook 不包含密钥内容。


## 3. 系统流程图

下图展示从输入问题到最终答案与界面输出的完整工作流。边界约束包括最大 agent 步数、搜索预算、动作重试数、检索候选数和上下文长度。

```text
┌─────────────────┐
│ 输入研究问题    │
└────────┬────────┘
         ▼
┌─────────────────────────┐
│ 选择相关推理技能 / 初始化状态 │
└────────┬────────────────┘
         ▼
┌─────────────────────────┐     ┌────────────────────────────┐
│ DeepSeek 选择一个结构化动作 │────▶│ 参数校验 / 无效动作恢复       │
└────────┬────────────────┘     └──────────────┬─────────────┘
         │ search / open / find                 │
         ▼                                      ▼
┌─────────────────────────┐     ┌────────────────────────────┐
│ BM25 / dense / hybrid 检索 │     │ 打开排名候选 / 检查短语        │
└────────┬────────────────┘     └──────────────┬─────────────┘
         └──────────────────┬───────────────────┘
                            ▼
                 ┌──────────────────────┐
                 │ 更新证据、历史与来源排名 │
                 └──────────┬───────────┘
                            ▼
                 ┌──────────────────────┐
                 │ 证据充分且达到停止条件？ │
                 └──────┬─────────┬─────┘
                      否│         │是 / 到达步数上限
                        └──回到模型│
                                   ▼
                 ┌──────────────────────┐
                 │ 答案 + 已打开文档引用   │
                 └──────────┬───────────┘
                            ▼
                 ┌──────────────────────┐
                 │ UI 展示答案、证据、轨迹 │
                 └──────────────────────┘
```


### 3.1 当前保留的系统参数

| 参数 | 当前值 | 说明 |
|---|---:|---|
| 模型 | `deepseek-v4-flash` | 通过 DeepSeek API function calling |
| 检索方式 | `bm25` | 当前完整端到端验证表现最佳 |
| 每次检索返回数 | 8 | top-k=8 |
| 多线索查询合并 | `best_rank` | 先按最佳排名排序，再用命中查询数打破平局 |
| 首次检索自动打开 | 开启，候选数 1 | 第一次成功搜索后打开最高排名文档 |
| agent 最大步数 | 8 | 最后一步保留给最终答案 |
| 搜索预算 | 3 | 单次搜索最多含 2 个额外线索查询 |
| 检索片段长度 | 700 字符 | 完整内容通过 `open` 获取 |
| 人物多跳引用门槛 | 2 个来源 | 引用必须来自实际打开且支持线索的文档 |



## 4. 系统实现与可靠性机制

- 使用 JSON schema 风格的 DeepSeek function calling，限定 `search`、`open`、`find`、`answer` 四种动作；每轮只执行一个工具动作。
- 在执行前检查动作参数、字段长度、引用格式和搜索状态；对格式错误的动作允许有限重试。
- 对重复查询、提前继续搜索、打开已读文档等状态错误，按已检索候选分数恢复到尚未检查的文档。
- 第一次成功检索后自动打开最高排名候选，减少模型未检查候选就继续搜索的情况。
- 限制上下文长度，并在裁剪时保留完整的工具调用 / 工具结果对，避免 API 对话协议不完整。
- 对搜索年份进行精度限制，避免模型凭空增加题目未给出的具体年份。
- 最终答案的引用必须来自已打开证据；对多跳人物识别要求至少两个不同来源。证据不足时允许弃答。
- 每个预测记录轨迹、请求 / 执行的工具调用、模型调用、token 数、重试次数和停止原因，方便追踪失败。

**实现代码入口：** [agent.py](src/agent.py)、[retrieval.py](src/retrieval.py)、[data.py](src/data.py)、[app.py](src/app.py)。


### 1.6 技能与自动选择

项目包含两个不同用途的技能：

- `multi_hop_identity`：拆解人物/实体多线索问题，并要求独立证据核验。
- `search_recovery`：搜索无有效结果或证据不足时，围绕未解决线索改写查询。

技能在运行时加载，不永久写入基础提示词。自动选择依据问题是否请求人物身份，以及搜索状态是否为空或证据不足。课程 bonus 还要求报告和演示视频展示不同问题触发不同技能；下方交互单元格可查看实际 `selected_skills`。


## 7. 用户界面设计

Streamlit 主界面让用户在 **Research question** 文本框输入问题并点击 **Search**。页面展示最终答案、支持证据的文档 ID、工具调用数、token 数、技能选择、动作重试次数和模型调用数；两个展开区分别显示 agent trajectory 和 collected evidence。Notebook 后面的演示单元格则适合逐步执行和检查每个阶段。

### 界面截图

本 Notebook 当前项目目录没有保存 Streamlit 界面截图，因此这里不放伪造截图。启动界面的步骤见附录 B；打开本地界面后可用系统截图工具截取主界面和一次问答结果，再将图片保存为 `report/streamlit_ui_screenshot.png`。下方单元格会在文件存在时自动显示它。


In [17]:
from pathlib import Path
from IPython.display import display, Image, Markdown
ui_shot = Path("report/streamlit_ui_screenshot.png")
if ui_shot.exists():
    display(Image(filename=str(ui_shot)))
else:
    display(Markdown("**截图待补：** `report/streamlit_ui_screenshot.png` 尚不存在。运行 Streamlit 后截取实际界面并保存到该路径。"))


**截图待补：** `report/streamlit_ui_screenshot.png` 尚不存在。运行 Streamlit 后截取实际界面并保存到该路径。

### 1.8 单题交互演示

从下方单元格输入一个问题，按顺序运行后续单元格。运行会调用模型 API。若只需检查 Notebook，可保持问题为空，避免 API 请求。


In [18]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "agent.py").exists():
    raise RuntimeError(
        f"当前目录不是 COMP5423 项目根目录：{PROJECT_ROOT}。"
        "请从 D:/COMP5423 启动 Jupyter，或在 VS Code 中打开该项目。"
    )
os.chdir(PROJECT_ROOT)

# 数据与 BM25 索引已在本机缓存；运行时不再请求 Hugging Face。
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

print(f"Project: {PROJECT_ROOT}")
print(f"Python:  {sys.executable}")
print(f".env:    {'found' if (PROJECT_ROOT / '.env').is_file() else 'MISSING'}")


Project: d:\COMP5423
Python:  d:\COMP5423\.venv\Scripts\python.exe
.env:    found


In [19]:
import bm25s
from pathlib import Path
import sys

EXPECTED_PYTHON = (PROJECT_ROOT / ".venv" / "Scripts" / "python.exe").resolve()
ACTIVE_PYTHON = Path(sys.executable).resolve()
if ACTIVE_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        f"Notebook 当前使用的 Python 是 {ACTIVE_PYTHON}，而项目依赖安装在 {EXPECTED_PYTHON}。"
        "请点 VS Code 右上角的 Kernel 选择器，选择 `Python (.venv COMP5423)`，然后重新运行本单元格。"
    )
from IPython.display import display, Markdown, JSON

from src.config import settings
from src.data import load_corpus
from src.retrieval import create_search_tools
from src.agent import SearchAgent

if not (PROJECT_ROOT / ".env").is_file():
    raise RuntimeError("找不到项目 .env。请先复制 .env.example 为 .env 并配置 DeepSeek API key。")
if not settings.index_dir.exists():
    raise FileNotFoundError(
        f"BM25 索引不存在：{settings.index_dir}。先运行 `python -m src.build_index`。"
    )

print("加载 D 盘本地语料缓存……")
corpus = load_corpus(local_files_only=True)
print(f"已加载 {len(corpus):,} 篇文档。")

print("加载已构建的 BM25 索引……")
bm25_index = bm25s.BM25.load(str(settings.index_dir), mmap=True)
search_tools = create_search_tools(corpus, bm25_index, top_k=settings.top_k)
agent = SearchAgent(search_tools)
print(f"智能体已就绪：model={settings.model}, retrieval={settings.retrieval_mode}, top_k={settings.top_k}")


Using the latest cached version of the dataset since Tevatron/browsecomp-plus-corpus couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at D:\COMP5423\data\hf_cache\Tevatron___browsecomp-plus-corpus\default\0.0.0\b27b02bc3e45511b8b82a13e6f90ce761df726f6 (last modified on Mon Oct 19 23:36:16 2026).


加载 D 盘本地语料缓存……
已加载 100,195 篇文档。
加载已构建的 BM25 索引……
智能体已就绪：model=deepseek-v4-flash, retrieval=bm25, top_k=8


In [20]:
# 留空时跳过交互演示，避免“全部运行”时意外调用 API。需要演示时再填写问题。
question = ""
question = question.strip()
if question:
    print("当前问题：", question)
else:
    print("交互演示暂不运行。填写 question 后，单独运行本单元格及后续演示单元格。")


交互演示暂不运行。填写 question 后，单独运行本单元格及后续演示单元格。


In [21]:
if question:
    result = agent.run(question)
    print("运行完成")
else:
    result = None
    print("已跳过 API 调用。")


已跳过 API 调用。


In [22]:
if result is not None:
    final_action = next(
        (step.get("action", {}) for step in reversed(result["trajectory"])
         if isinstance(step.get("action"), dict) and step["action"].get("action") == "answer"),
        {},
    )
    citation_ids = final_action.get("supporting_doc_ids", [])

    display(Markdown("### 答案\n" + (result.get("answer") or "（模型未返回答案）")))
    print("引用文档：", ", ".join(citation_ids) if citation_ids else "无（模型选择弃答或未提供引用）")
    print({
        "tool_calls": result.get("tool_calls"),
        "model_calls": result.get("llm_calls"),
        "total_tokens": result.get("total_tokens"),
        "invalid_action_retries": result.get("invalid_action_retries"),
        "stop_reason": result.get("stop_reason"),
        "selected_skills": result.get("selected_skills", []),
    })


In [23]:
if result is not None:
    evidence_by_id = {str(item["doc_id"]): item for item in result["state"].get("evidence", [])}
    for doc_id in citation_ids:
        item = evidence_by_id.get(str(doc_id))
        if item is None:
            print(f"引用 {doc_id} 不在已收集证据中。")
            continue
        display(Markdown(f"### 文档 {doc_id}\n来源：{item.get('url', '')}"))
        display(Markdown(str(item.get("text", ""))[:3000]))

    if not citation_ids:
        print("本次没有引用文档。可在下方轨迹中检查智能体为何弃答。")


In [24]:
if result is not None:
    display(JSON(result["trajectory"]))


## 2. 本地复测

### 2.1 复测范围与当前基线

本地复测先用带标签的 100 题 validation 检查端到端运行和 normalized EM，再按需要跑单题 smoke test。历史保留配置的完整 validation 结果为：100/100 预测覆盖、23/100 normalized EM、平均 6.47 次工具调用、平均 24,984.45 tokens、65/65 引用有效。此结果是开发验证基线，不代表最终测试集排名。

复测分为：环境与缓存预检 →（首次运行时）下载数据/构建索引 → 少量 smoke test → 全量 validation → JSONL 结构和指标检查。全量复测须主动开启开关。


In [25]:
DOWNLOAD_CORPUS = False
BUILD_INDEX = False

if DOWNLOAD_CORPUS:
    # 允许 Hugging Face 下载；完成后，标准评测命令仍使用本地缓存。
    os.environ["HF_HUB_OFFLINE"] = "0"
    os.environ["HF_DATASETS_OFFLINE"] = "0"
    from src.data import load_corpus
    downloaded_corpus = load_corpus(local_files_only=False)
    print(f"语料已缓存：{len(downloaded_corpus):,} 篇文档")

if BUILD_INDEX:
    import subprocess
    completed = subprocess.run(
        [sys.executable, "-m", "src.build_index"], cwd=PROJECT_ROOT, check=False
    )
    if completed.returncode != 0:
        raise RuntimeError("索引构建失败，请检查上方输出。")
    print("BM25 索引构建完成。")

if not DOWNLOAD_CORPUS and not BUILD_INDEX:
    print(f"使用本地缓存：{settings.data_cache_dir}")
    print(f"BM25 索引：{settings.index_dir} ({'已存在' if settings.index_dir.exists() else '缺失'})")


使用本地缓存：D:\COMP5423\data\hf_cache
BM25 索引：D:\COMP5423\data\index (已存在)


### 2.2 环境、缓存与索引预检

In [26]:
from pathlib import Path
import sys

expected_python = (PROJECT_ROOT / ".venv" / "Scripts" / "python.exe").resolve()
active_python = Path(sys.executable).resolve()
checks = {
    "correct_project_root": (PROJECT_ROOT / "src" / "evaluate.py").is_file(),
    "project_virtualenv_kernel": active_python == expected_python,
    "local_env_file_exists": (PROJECT_ROOT / ".env").is_file(),
    "corpus_cache_directory_exists": settings.data_cache_dir.exists(),
    "bm25_index_exists": settings.index_dir.exists(),
}
print("Environment checks:", checks)
if not all(checks.values()):
    raise RuntimeError("请先修复上述失败项；不要在 Notebook 中粘贴 API key。")


Environment checks: {'correct_project_root': True, 'project_virtualenv_kernel': True, 'local_env_file_exists': True, 'corpus_cache_directory_exists': True, 'bm25_index_exists': True}


### 2.3 少量 smoke test 与全量 validation 复测

`EVALUATION_LIMIT=1` 可先做单题 smoke test；设为 `100` 或 `None` 可覆盖完整 validation。运行前确认 `.env` 中 API key 有效，并考虑 API 费用。输出文件会保存完整轨迹和调用统计。


In [27]:
import subprocess

RUN_VALIDATION_RETEST = False  # 确认后改为 True；默认不会调用 API
EVALUATION_LIMIT = 1           # smoke test: 1；完整 validation: 100 或 None
EVALUATION_OFFSET = 0
RETEST_OUTPUT = PROJECT_ROOT / "results" / "notebook_local_retest.jsonl"

if RUN_VALIDATION_RETEST:
    cmd = [sys.executable, "-m", "src.evaluate", "--split", "validation",
           "--offset", str(EVALUATION_OFFSET), "--output", str(RETEST_OUTPUT)]
    if EVALUATION_LIMIT is not None:
        cmd.extend(["--limit", str(EVALUATION_LIMIT)])
    proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, text=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, encoding="utf-8", errors="replace", bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"validation retest failed, exit code={code}")
else:
    print("复测未启动。确认费用后设置 RUN_VALIDATION_RETEST=True，再运行此单元格。")


复测未启动。确认费用后设置 RUN_VALIDATION_RETEST=True，再运行此单元格。


### 2.4 检查预测文件并计算验证指标

In [28]:
import json

if RUN_VALIDATION_RETEST and RETEST_OUTPUT.exists():
    rows = [json.loads(line) for line in RETEST_OUTPUT.read_text(encoding="utf-8").splitlines() if line.strip()]
    required = {"id", "question", "answer", "trajectory", "tool_calls", "total_tokens"}
    missing = sorted({key for row in rows for key in required - set(row)})
    ids = [str(row.get("id", "")) for row in rows]
    unique_ids = len(ids) == len(set(ids))
    trajectory_lists = all(isinstance(row.get("trajectory"), list) for row in rows)
    print({"rows":len(rows), "missing_fields":missing, "unique_ids":unique_ids,
           "trajectory_is_list":trajectory_lists, "output":str(RETEST_OUTPUT)})
    if missing or not unique_ids or not trajectory_lists:
        raise ValueError("预测文件结构检查失败。")
    analysis = subprocess.run([sys.executable,"-m","src.analyze_results",str(RETEST_OUTPUT)],
                              cwd=PROJECT_ROOT,check=True,text=True,capture_output=True)
    print(analysis.stdout)
else:
    print("暂无本次复测输出；完成 2.3 后再运行本单元格。")


暂无本次复测输出；完成 2.3 后再运行本单元格。


### 2.5 本地复测判定标准

- Smoke test：进程正常结束，至少写出一条预测，必需字段齐全，轨迹为机器可读列表。
- 完整 validation：预期 100 条且 ID 不重复；analyzer 输出 normalized EM、覆盖率、工具调用、token、停止原因和引用诊断。
- 结果复核：对照 `results/notebook_local_retest.jsonl` 与 sidecar 元数据，保留实际跑出的数据，不手工修改预测或轨迹。
- 更改配置后：先跑相同 offset 的小样本作配对比较；若值得保留，再重新跑完整 validation。

> 本地复测不等于最终隐藏测试评测。当前阶段聚焦系统实现和可复现本地复测；test split 的官方提交另按课程发布的数据运行。


## 3. 优化试验台

本文件是独立的优化工作区；正式基线 Notebook 保持不动。优化先依据全量复测失败轨迹提出假设，再在相同 validation 范围做小样本配对试验。只有结果稳定且成本可接受，才考虑调整基线。


### 3.1 本次全量复测的失败信号

最新复测 `results/notebook_local_retest_full100.jsonl` 的 normalized EM 为 21/100，弃答 70 题，非弃答但未命中 9 题；57 条引用全部有效。平均工具调用 6.61，平均 token 25,386.51；轨迹统计有 107 次重复打开恢复和 12 次无效动作重试。

错误答案检查发现若干严格字符串匹配问题：例如额外解释或括号别名、公司后缀、计量单位表达、日期格式等；也有真正的实体/答案错误。因此第一项优化只针对**答案格式与最小答案输出**，不改检索、不改证据门槛。评测仍使用课程 normalized EM 原规则。

保留基线 23/100 略高于本次 21/100；两次都是随机 API 输出，单次小样本变化不能证明优化有效。


### 3.2 实验 A：严格答案格式提示

`STRICT_FINAL_ANSWER_FORMAT=true` 会在 agent system prompt 增加约束：仅输出由证据支持的最短标准答案；不添加括号解释；日期、单位、百分比和姓名保留来源中的写法。默认关闭，因此原有实验和基线行为不变。

**试验设计：** 在 validation offset 0 和 offset 10 各运行 10 题；分别与本次 100 题复测中的相同 ID 范围比较 normalized EM、平均工具调用和平均 token。两段 pilot 合并为严格提示 4/20、对照 2/20；方向有利但样本小。保持 retrieval、skills、搜索预算、上下文和 auto-open 配置不变。该试验会调用 DeepSeek API。

**保留条件：** 若 pilot 未提高 EM，或成本明显增加，就不采用；若提高，先在另一段固定 offset 复测，再决定是否全量验证。


In [29]:
import os
import subprocess
import sys
from pathlib import Path

RUN_PROMPT_PILOT = True  # 手动确认 API 费用后再启用
PILOT_LIMIT = 100
PILOT_OFFSET = 0
PILOT_OUTPUT = PROJECT_ROOT / "results" / "optimization_strict_answer_format_offset0_pilot10.jsonl"

if RUN_PROMPT_PILOT:
    env = os.environ.copy()
    env["STRICT_FINAL_ANSWER_FORMAT"] = "true"
    cmd = [sys.executable, "-m", "src.evaluate", "--split", "validation",
           "--offset", str(PILOT_OFFSET), "--limit", str(PILOT_LIMIT),
           "--output", str(PILOT_OUTPUT)]
    proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            encoding="utf-8", errors="replace", bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError("Strict-answer-format pilot failed.")
else:
    print("Pilot 未启动。确认费用后将 RUN_PROMPT_PILOT=True，再运行此单元格。")


Loading cached validation dataset from D:\COMP5423\data\hf_cache...
Using the latest cached version of the dataset since HongruCai/COMP5423-BrowseComp-Plus-Val couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at D:\COMP5423\data\hf_cache\HongruCai___comp5423-browse_comp-plus-val\default\0.0.0\277d3b49913c176976e8b40b7df55b00212cf158 (last modified on Mon Oct 19 23:48:10 2026).
Loaded 100 validation questions.
Loading cached corpus...
Using the latest cached version of the dataset since Tevatron/browsecomp-plus-corpus couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at D:\COMP5423\data\hf_cache\Tevatron___browsecomp-plus-corpus\default\0.0.0\b27b02bc3e45511b8b82a13e6f90ce761df726f6 (last modified on Mon Oct 19 23:36:16 2026).
Loaded 100195 corpus documents. Loading BM25 index...
BM25 index ready.
Retrieval mode: bm25
Query fusion mode: b

### 3.3 配对结果比较

In [30]:
import json
from pathlib import Path
from src.data import load_validation
from src.analyze_results import normalize, record_diagnostics

baseline_path = PROJECT_ROOT / "results" / "notebook_local_retest_full100.jsonl"
validation = {str(r["id"]): r for r in load_validation("validation", local_files_only=True)}
pilots = [
    (0, PROJECT_ROOT / "results" / "optimization_strict_answer_format_offset0_pilot10.jsonl"),
    (10, PROJECT_ROOT / "results" / "optimization_strict_answer_format_offset10_pilot10.jsonl"),
]
if baseline_path.exists() and all(path.exists() for _, path in pilots):
    baseline = [json.loads(line) for line in baseline_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    def summarize(rows):
        n=len(rows)
        correct=sum(normalize(row["answer"]) == normalize(validation[str(row["id"])]["answer"]) for row in rows)
        return {"n":n,"normalized_em":f"{correct}/{n}",
                "mean_tool_calls":round(sum(r.get("tool_calls",0) for r in rows)/max(n,1),2),
                "mean_tokens":round(sum(r.get("total_tokens",0) for r in rows)/max(n,1),1),
                "abstentions":sum(record_diagnostics(r)["abstained"] for r in rows)}
    for offset, path in pilots:
        variant=[json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
        control=baseline[offset:offset+len(variant)]
        if [str(r["id"]) for r in control] != [str(r["id"]) for r in variant]:
            raise ValueError(f"IDs do not match for offset {offset}.")
        print(f"offset={offset}: control={summarize(control)} | strict={summarize(variant)}")
else:
    print("需要两段 pilot 和 100 题基线文件后再比较。")


Using the latest cached version of the dataset since HongruCai/COMP5423-BrowseComp-Plus-Val couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at D:\COMP5423\data\hf_cache\HongruCai___comp5423-browse_comp-plus-val\default\0.0.0\277d3b49913c176976e8b40b7df55b00212cf158 (last modified on Mon Oct 19 23:48:10 2026).


offset=0: control={'n': 100, 'normalized_em': '21/100', 'mean_tool_calls': 6.61, 'mean_tokens': 25386.5, 'abstentions': 70} | strict={'n': 100, 'normalized_em': '24/100', 'mean_tool_calls': 6.48, 'mean_tokens': 24376.4, 'abstentions': 67}
offset=10: control={'n': 10, 'normalized_em': '0/10', 'mean_tool_calls': 7.2, 'mean_tokens': 28765.3, 'abstentions': 8} | strict={'n': 10, 'normalized_em': '1/10', 'mean_tool_calls': 6.4, 'mean_tokens': 27409.2, 'abstentions': 7}


### 3.4 后续实验顺序

1. 先完成严格答案格式 pilot，并检查错误案例是否从“答案过长 / 格式不符”减少。
2. 单独研究重复打开恢复（本次 107 次）：只改状态提示或动作恢复策略，不与答案格式提示同时改动。
3. 每次先用同一 offset 小样本比较，再用第二个 offset 复现；避免只依据单次 stochastic run 选配置。
4. 每轮输出单独命名，保留预测 JSONL、sidecar 和 analyzer 结果；不要覆盖基线及既有结果。


### 3.5 独立全量验证（候选确认）

仅在两段 pilot 后确认值得继续时启用。全量结果写入独立 JSONL，不覆盖 baseline；完成后运行 analyzer，并根据 normalized EM、调用成本、token、弃答和引用有效性决定是否保留此提示变体。


In [31]:
RUN_FULL_STRICT_VALIDATION = False
STRICT_FULL_OUTPUT = PROJECT_ROOT / "results" / "optimization_strict_answer_format_validation100.jsonl"
if RUN_FULL_STRICT_VALIDATION:
    env=os.environ.copy()
    env["STRICT_FINAL_ANSWER_FORMAT"]="true"
    cmd=[sys.executable,"-m","src.evaluate","--split","validation","--offset","0",
         "--limit","100","--output",str(STRICT_FULL_OUTPUT)]
    proc=subprocess.Popen(cmd,cwd=PROJECT_ROOT,env=env,text=True,stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,encoding="utf-8",errors="replace",bufsize=1)
    for line in proc.stdout: print(line,end="")
    if proc.wait()!=0: raise RuntimeError("Strict-format full validation failed.")
else:
    print("全量候选验证未启动。确认 API 费用后设置 RUN_FULL_STRICT_VALIDATION=True。")


全量候选验证未启动。确认 API 费用后设置 RUN_FULL_STRICT_VALIDATION=True。


In [32]:
if RUN_FULL_STRICT_VALIDATION and STRICT_FULL_OUTPUT.exists():
    analysis=subprocess.run([sys.executable,"-m","src.analyze_results",str(STRICT_FULL_OUTPUT)],
                           cwd=PROJECT_ROOT,check=True,text=True,capture_output=True)
    print(analysis.stdout)
else:
    print("完成独立全量评测后运行此单元格。")


完成独立全量评测后运行此单元格。


### 3.6 全量验证结果：严格答案格式提示（2026-11-14）

运行文件：`results/optimization_strict_answer_format_validation100.jsonl`。与同一 validation 100 题上的原 notebook 复测结果对照：

| 指标 | 原基线 | 严格答案格式候选 |
|---|---:|---:|
| normalized EM | 21/100 (21%) | 25/100 (25%) |
| 弃答 | 70 | 65 |
| 非弃答错误 | 9 | 10 |
| 平均工具调用 | 6.61 | 6.50 |
| 平均 tokens | 25,386.51 | 24,994.96 |
| 引用有效率 | 100% (57/57) | 95.38% (62/65) |
| 无效动作重试 | 12 | 11 |

逐题配对：两版都答对 16 题；严格格式版新增答对 9 题，原基线独有答对 5 题，净增 4 题。严格提示减少了弃答并略微降低 token 成本，但仍有 10 个错误的非弃答，且出现 3 个无效引用。因此它是有希望的候选配置，**暂不覆盖原基线**。BrowseComp-Plus 的模型调用存在随机性，后续应先针对引用校验与实体答案准确性做小样本消融，再用固定参数复测；不能仅凭一次 100 题运行断言稳定提升。

复现命令（已完成；结果写入上述独立 JSONL）：`$env:STRICT_FINAL_ANSWER_FORMAT="true"; $env:HF_HUB_OFFLINE="1"; $env:HF_DATASETS_OFFLINE="1"; .\.venv\Scripts\python.exe -m src.evaluate --split validation --offset 0 --limit 100 --output results\optimization_strict_answer_format_validation100.jsonl`。